# Data preprocessing

Combine raw CSV, XLSX, or Parquet files without discarding missing feature or label values.

In [ ]:
from pathlib import Path

import pandas as pd

In [ ]:
ROOT = Path.cwd().resolve()
if not (ROOT / "data").exists():
    ROOT = ROOT.parent
if not (ROOT / "data").exists():
    raise FileNotFoundError("Run this notebook from the repository or notebooks directory")

raw_dir = ROOT / "data/raw"
output_path = ROOT / "data/preprocessed/dataset.parquet"
raw_files = sorted(
    path for path in raw_dir.iterdir() if path.is_file() and not path.name.startswith(".")
)
if not raw_files:
    raise FileNotFoundError(f"No raw files found in {raw_dir}")
raw_files

In [ ]:
MISSING_TOKENS = ["", "NA", "N/A", "null", "NULL", "None"]

def read_raw(path):
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path, na_values=MISSING_TOKENS, keep_default_na=True)
    if path.suffix.lower() == ".xlsx":
        return pd.read_excel(path, na_values=MISSING_TOKENS)
    if path.suffix.lower() in {".parquet", ".pq"}:
        return pd.read_parquet(path)
    raise ValueError(f"Unsupported raw file: {path.name}")

data = pd.concat([read_raw(path) for path in raw_files], ignore_index=True, sort=False)
data = data.replace(MISSING_TOKENS, pd.NA)
data.head()

In [ ]:
output_path.parent.mkdir(parents=True, exist_ok=True)
data.to_parquet(output_path, index=False)
missing_counts = data.isna().sum()
print(f"Wrote {len(data)} rows and {len(data.columns)} columns to {output_path}")
display(missing_counts[missing_counts > 0].rename("missing_count"))